In [2]:
import chromadb
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

# ChromaDB built-in embedding — no torch needed
client = chromadb.PersistentClient(path="../rag/vectorstore/chroma_db")

# Delete existing collection if exists
try:
    client.delete_collection("jobs")
except:
    pass

# Create new collection
collection = client.create_collection("jobs")

# Load clean data
df = pd.read_csv("../data/processed/jobs_cleaned.csv")
df = df.head(500)

# Prepare texts and IDs
texts = []
ids = []
metadatas = []

for i, row in df.iterrows():
    text = f"Job Title: {row['title']}. Company: {row['company_name']}. Location: {row['location']}. Skills: {row['skills']}"
    texts.append(text)
    ids.append(str(i))
    metadatas.append({
        "title": str(row['title']),
        "company": str(row['company_name']),
        "location": str(row['location'])
    })

# Add to ChromaDB
collection.add(
    documents=texts,
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Done! Stored {collection.count()} jobs in ChromaDB")

/Users/macbookpro/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:31<00:00, 907kiB/s] 


✅ Done! Stored 500 jobs in ChromaDB


In [7]:
import requests
import os

def ask_question(question):
    
    # Step 1 — search ChromaDB for relevant jobs
    results = collection.query(
        query_texts=[question],
        n_results=5
    )
    
    # Step 2 — build context from results
    context = "\n".join(results['documents'][0])
    
    # Step 3 — send to Ollama llama3.2 locally
    prompt = f"""You are an expert UAE job market analyst.

You have access to this real job market data:
{context}

Answer this question: {question}

Format your answer as:
- Start with a direct one-sentence answer
- Then provide 3-5 specific points with details
- End with one practical recommendation

Important: Use ONLY the job data provided above.
If information is not in the data, say so clearly."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:1b",
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]

# Ask your first question!
question = "What skills are most needed for data analyst jobs?"
print(f"Q: {question}")
print(f"\nA: {ask_question(question)}")

Q: What skills are most needed for data analyst jobs?

A: Based on the provided job market data, the most needed skills for data analyst jobs are:

- Excel
- SQL
- SAS

These skills are consistently present across multiple job postings, indicating their high demand in the UAE and US markets.

Here are specific details about these skills:
 
- **Excel**: As seen in Job Title: Business / Data Analyst with National Center On Institutions And., Excel is a widely used tool for data analysis, data visualization, and reporting.
- **SQL**: The job postings also highlight SQL as a crucial skill, which is essential for querying, analyzing, and manipulating large datasets in various industries.
- **SAS**: Many job postings mention SAS as a required skillset, particularly in the Senior Data Analyst role at WS Development, indicating its importance in data analysis and processing.

Here's a practical recommendation:
- Ensure proficiency in Excel to perform basic data analysis tasks such as data clea

In [6]:
questions = [
    "Which companies hire the most data analysts?",
    "What locations have the most data analyst jobs?",
    "What is the salary range for data analyst jobs?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_question(q)}")
    print("-" * 50)


Q: Which companies hire the most data analysts?
A: Based on the provided job data, Walmart and Target Brands, Inc. appear to be the two companies that hire the most data analysts, with 2 positions each. 

In fact, there are three different titles for Data Analyst in these companies:

- One title at Walmart is Analyst II, Data Analytics.
- Two titles at Target Brands, Inc. are Sr Data Analyst and Data Analyst.

Therefore, I can confidently say that both Walmart and Target Brands, Inc. hire the most data analysts among all the job postings provided.
--------------------------------------------------

Q: What locations have the most data analyst jobs?
A: Based on the provided job data, the location with the most data analyst jobs is the City of New York. There are three separate instances of Data Analyst positions:

- One position at NYC Kids RISE (United States).
- Two positions at the Modern Technology Solutions Inc. (United States), and
- Three positions at the City of New York (Unite

In [9]:
import chromadb
import pandas as pd

client = chromadb.PersistentClient(path="../rag/vectorstore/chroma_db")

try:
    client.delete_collection("jobs")
except:
    pass

collection = client.create_collection("jobs")

df = pd.read_csv("../data/processed/jobs_cleaned.csv")

texts = []
ids = []
metadatas = []

for i, row in df.iterrows():
    text = f"Job Title: {row['title']}. Company: {row['company_name']}. Location: {row['location']}. Skills: {row['skills']}. Salary: {row['salary_standardized']}"
    texts.append(text)
    ids.append(str(i))
    metadatas.append({
        "title": str(row['title']),
        "company": str(row['company_name']),
        "location": str(row['location'])
    })

batch_size = 1000
for i in range(0, len(texts), batch_size):
    collection.add(
        documents=texts[i:i+batch_size],
        ids=ids[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size]
    )
    print(f"Added {min(i+batch_size, len(texts))}/{len(texts)} jobs...")

print(f"\n✅ Done! Total stored: {collection.count()} jobs")

Added 1000/29621 jobs...
Added 2000/29621 jobs...
Added 3000/29621 jobs...
Added 4000/29621 jobs...
Added 5000/29621 jobs...
Added 6000/29621 jobs...
Added 7000/29621 jobs...
Added 8000/29621 jobs...
Added 9000/29621 jobs...
Added 10000/29621 jobs...
Added 11000/29621 jobs...
Added 12000/29621 jobs...
Added 13000/29621 jobs...
Added 14000/29621 jobs...
Added 15000/29621 jobs...
Added 16000/29621 jobs...
Added 17000/29621 jobs...
Added 18000/29621 jobs...
Added 19000/29621 jobs...
Added 20000/29621 jobs...
Added 21000/29621 jobs...
Added 22000/29621 jobs...
Added 23000/29621 jobs...
Added 24000/29621 jobs...
Added 25000/29621 jobs...
Added 26000/29621 jobs...
Added 27000/29621 jobs...
Added 28000/29621 jobs...
Added 29000/29621 jobs...
Added 29621/29621 jobs...

✅ Done! Total stored: 29621 jobs


In [10]:
import pandas as pd

df = pd.read_csv("../data/processed/jobs_cleaned.csv")
df_sample = df.head(5000)
df_sample.to_csv("../data/processed/jobs_sample.csv", index=False)

print(f"Sample saved: {len(df_sample)} rows")
print(f"File size is now small enough for GitHub!")

Sample saved: 5000 rows
File size is now small enough for GitHub!
